# CORnet-S recurrence ablation (Kubilius et al. 2019, Fig. 4) on Colab
**Before running:** Runtime → Change runtime type → GPU (T4 is fine; A100/L4 is faster).
Run the cells top to bottom. If Colab disconnects, re-run everything: training resumes from the checkpoints saved in Google Drive.

Protocol used here is a **reduced** one (Tiny-ImageNet, 64×64, 200 classes) because full ImageNet does not fit Colab's disk/CPU limits. State this in your write-up.

In [ ]:
import os, subprocess, torch
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cornet_repro'
RUNS  = f'{DRIVE}/runs'
os.makedirs(RUNS, exist_ok=True); os.makedirs('/content/code', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - change the runtime type!')

## 1. Write the code files to /content/code

In [ ]:
%%writefile /content/code/cornet_s.py
"""CORnet-S with configurable recurrent steps per area.

Re-implemented to mirror the official dicarlolab/cornet `cornet_s.py`
(parameter/module names are kept identical so official pretrained weights load
with strict key matching when times=(2, 4, 2)). Diff this file against the
official one before trusting results -- eval_pretrained_sweep.py checks it end to end.
"""
import math
from collections import OrderedDict

import torch
from torch import nn


class Flatten(nn.Module):
    def forward(self, x):
        return x.view(x.size(0), -1)


class Identity(nn.Module):
    def forward(self, x):
        return x


class CORblock_S(nn.Module):
    scale = 4  # bottleneck expansion

    def __init__(self, in_channels, out_channels, times=1):
        super().__init__()
        self.times = times
        self.conv_input = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.skip = nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=2, bias=False)
        self.norm_skip = nn.BatchNorm2d(out_channels)

        self.conv1 = nn.Conv2d(out_channels, out_channels * self.scale, kernel_size=1, bias=False)
        self.nonlin1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels * self.scale, out_channels * self.scale,
                               kernel_size=3, stride=2, padding=1, bias=False)
        self.nonlin2 = nn.ReLU(inplace=True)
        self.conv3 = nn.Conv2d(out_channels * self.scale, out_channels, kernel_size=1, bias=False)
        self.nonlin3 = nn.ReLU(inplace=True)
        self.output = Identity()

        # BatchNorm is NOT shared over time steps (paper, Sec. 2.1)
        for t in range(self.times):
            setattr(self, f'norm1_{t}', nn.BatchNorm2d(out_channels * self.scale))
            setattr(self, f'norm2_{t}', nn.BatchNorm2d(out_channels * self.scale))
            setattr(self, f'norm3_{t}', nn.BatchNorm2d(out_channels))

    def forward(self, inp):
        x = self.conv_input(inp)
        for t in range(self.times):
            if t == 0:
                skip = self.norm_skip(self.skip(x))
                self.conv2.stride = (2, 2)
            else:
                skip = x
                self.conv2.stride = (1, 1)

            x = self.conv1(x)
            x = getattr(self, f'norm1_{t}')(x)
            x = self.nonlin1(x)

            x = self.conv2(x)
            x = getattr(self, f'norm2_{t}')(x)
            x = self.nonlin2(x)

            x = self.conv3(x)
            x = getattr(self, f'norm3_{t}')(x)

            x = x + skip
            x = self.nonlin3(x)
            output = self.output(x)
        return output


def CORnet_S(times=(2, 4, 2), num_classes=1000):
    """times = (V2 steps, V4 steps, IT steps). Paper baseline is (2, 4, 2)."""
    v2, v4, it = times
    model = nn.Sequential(OrderedDict([
        ('V1', nn.Sequential(OrderedDict([
            ('conv1', nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)),
            ('norm1', nn.BatchNorm2d(64)),
            ('nonlin1', nn.ReLU(inplace=True)),
            ('pool', nn.MaxPool2d(kernel_size=3, stride=2, padding=1)),
            ('conv2', nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False)),
            ('norm2', nn.BatchNorm2d(64)),
            ('nonlin2', nn.ReLU(inplace=True)),
            ('output', Identity()),
        ]))),
        ('V2', CORblock_S(64, 128, times=v2)),
        ('V4', CORblock_S(128, 256, times=v4)),
        ('IT', CORblock_S(256, 512, times=it)),
        ('decoder', nn.Sequential(OrderedDict([
            ('avgpool', nn.AdaptiveAvgPool2d(1)),
            ('flatten', Flatten()),
            ('linear', nn.Linear(512, num_classes)),
            ('output', Identity()),
        ]))),
    ]))

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            m.weight.data.normal_(0, math.sqrt(2.0 / n))
        elif isinstance(m, nn.BatchNorm2d):
            m.weight.data.fill_(1)
            m.bias.data.zero_()
    return model


def count_params(model):
    return sum(p.numel() for p in model.parameters())


if __name__ == '__main__':
    for cfg in [(1, 1, 1), (2, 2, 2), (2, 4, 2), (4, 4, 4)]:
        m = CORnet_S(cfg)
        y = m(torch.randn(2, 3, 224, 224))
        print(cfg, tuple(y.shape), f'{count_params(m)/1e6:.2f}M params')

In [ ]:
%%writefile /content/code/train.py
"""Train one CORnet-S variant on ImageNet (or a class subset) with the paper's recipe.

Paper (Sec. 2.2): SGD momentum 0.9, batch 256, lr 0.1 divided by 10 every 20
epochs, 43 epochs, cross-entropy. Weight decay is not stated in the paper; 1e-4
is the value I recall from the official run.py -- verify and change if needed.

Data layout: <data>/train/<class>/*.JPEG and <data>/val/<class>/*.JPEG
(the raw ImageNet val set must first be sorted into class folders).
"""
import argparse
import json
import os
import random
import time

import numpy as np
import torch
from torch import nn
from torchvision import datasets, transforms

from cornet_s import CORnet_S, count_params


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True)
    p.add_argument('--times', type=int, nargs=3, default=[2, 4, 2], help='V2 V4 IT steps')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--epochs', type=int, default=43)
    p.add_argument('--lr', type=float, default=0.1)
    p.add_argument('--step-size', type=int, default=20)
    p.add_argument('--bs', type=int, default=256)
    p.add_argument('--wd', type=float, default=1e-4)
    p.add_argument('--workers', type=int, default=8)
    p.add_argument('--n-classes', type=int, default=0, help='0 = all 1000; else use every k-th class')
    p.add_argument('--img-size', type=int, default=224, help='crop size (use e.g. 128 for a cheaper run)')
    p.add_argument('--max-train-per-class', type=int, default=0, help='0 = all images')
    p.add_argument('--amp', action='store_true', help='mixed precision (faster; note in your report)')
    p.add_argument('--min-scale', type=float, default=0.08, help='RandomResizedCrop min scale (0.35 is saner at 64px)')
    p.add_argument('--ckpt-every', type=int, default=1, help='save a resume checkpoint every N epochs')
    p.add_argument('--out', default='runs')
    p.add_argument('--name', default=None)
    return p.parse_args()


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def make_dataset(root, tf, n_classes, max_per_class=0):
    ds = datasets.ImageFolder(root, tf)
    if n_classes and n_classes < len(ds.classes):
        k = len(ds.classes) // n_classes
        keep = list(range(0, len(ds.classes), k))[:n_classes]
        remap = {old: new for new, old in enumerate(keep)}
        samples = [(p, remap[c]) for p, c in ds.samples if c in remap]
        ds.classes = [ds.classes[i] for i in keep]
        ds.class_to_idx = {c: i for i, c in enumerate(ds.classes)}
        ds.samples = samples
    if max_per_class:
        counts, kept = {}, []
        for p, c in ds.samples:
            if counts.get(c, 0) < max_per_class:
                kept.append((p, c)); counts[c] = counts.get(c, 0) + 1
        ds.samples = kept
    ds.imgs = ds.samples
    ds.targets = [c for _, c in ds.samples]
    return ds


@torch.no_grad()
def evaluate(model, loader, device, amp):
    model.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', enabled=amp):
            out = model(x)
        top5 = out.topk(5, dim=1).indices
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
        n += y.numel()
    return 100.0 * c1 / n, 100.0 * c5 / n


def main():
    a = get_args()
    seed_all(a.seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    name = a.name or f"cornet_s_{a.times[0]}-{a.times[1]}-{a.times[2]}_seed{a.seed}"
    os.makedirs(a.out, exist_ok=True)
    ckpt_path = os.path.join(a.out, name + '.ckpt')
    res_path = os.path.join(a.out, name + '.json')

    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    s = a.img_size
    train_tf = transforms.Compose([transforms.RandomResizedCrop(s, scale=(a.min_scale, 1.0)), transforms.RandomHorizontalFlip(),
                                   transforms.ToTensor(), norm])
    val_tf = transforms.Compose([transforms.Resize(int(round(s * 256 / 224))), transforms.CenterCrop(s),
                                 transforms.ToTensor(), norm])
    train_ds = make_dataset(os.path.join(a.data, 'train'), train_tf, a.n_classes, a.max_train_per_class)
    val_ds = make_dataset(os.path.join(a.data, 'val'), val_tf, a.n_classes)
    n_classes = len(train_ds.classes)
    train_dl = torch.utils.data.DataLoader(train_ds, a.bs, shuffle=True, num_workers=a.workers,
                                           pin_memory=True, drop_last=True,
                                           persistent_workers=a.workers > 0)
    val_dl = torch.utils.data.DataLoader(val_ds, a.bs, shuffle=False, num_workers=a.workers, pin_memory=True)

    model = CORnet_S(tuple(a.times), num_classes=n_classes).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=a.lr, momentum=0.9, weight_decay=a.wd)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=a.step_size, gamma=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=a.amp and device == 'cuda')
    crit = nn.CrossEntropyLoss()

    history, start = [], 0
    if os.path.exists(ckpt_path):  # resume
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); history = ck['history']; start = ck['epoch'] + 1
        print(f'resumed {name} from epoch {start}')

    print(f'{name}: {count_params(model)/1e6:.2f}M params, {len(train_ds)} train / {len(val_ds)} val images, '
          f'{n_classes} classes, device={device}')
    for ep in range(start, a.epochs):
        model.train(); t0 = time.time(); tot = nb = 0
        for x, y in train_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=a.amp and device == 'cuda'):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); nb += 1
        sched.step()
        top1, top5 = evaluate(model, val_dl, device, a.amp and device == 'cuda')
        history.append(dict(epoch=ep, train_loss=tot / nb, val_top1=top1, val_top5=top5, secs=time.time() - t0))
        print(f'[{name}] ep {ep+1}/{a.epochs} loss {tot/nb:.3f} top1 {top1:.2f} top5 {top5:.2f} ({time.time()-t0:.0f}s)')
        if (ep + 1) % a.ckpt_every == 0 or ep + 1 == a.epochs:
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                            history=history, epoch=ep), ckpt_path)

    final = history[-1]
    json.dump(dict(name=name, times=a.times, seed=a.seed, n_classes=n_classes, img_size=a.img_size,
                   epochs=a.epochs, params=count_params(model), final_top1=final['val_top1'],
                   final_top5=final['val_top5'], args=vars(a), history=history),
              open(res_path, 'w'), indent=1)
    print('final top-1 (last epoch, no best-epoch selection):', final['val_top1'])


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/code/run_all.py
"""Run the Figure-4 recurrence ablation: train each config x seed sequentially.
Edit CONFIGS/SEEDS or pass extra flags (they are forwarded to train.py), e.g.
  python run_all.py --data D:/imagenet --workers 8 --amp
  python run_all.py --data D:/imagenet --n-classes 100 --amp          # reduced: ImageNet-100
"""
import subprocess
import sys

CONFIGS = [(2, 4, 2), (1, 1, 1), (2, 2, 2), (4, 4, 4)]   # baseline first
SEEDS = [0]                                              # add 1, 2 if compute allows

for seed in SEEDS:
    for cfg in CONFIGS:
        cmd = [sys.executable, 'train.py', '--times', *map(str, cfg), '--seed', str(seed), *sys.argv[1:]]
        print('>>', ' '.join(cmd), flush=True)
        subprocess.run(cmd, check=True)   # train.py resumes automatically if interrupted

In [ ]:
%%writefile /content/code/plot_results.py
"""Aggregate runs/*.json -> results table + plot of delta top-1 vs recurrence config,
overlaid with the deltas read off the paper's Figure 4 (row 1)."""
import glob
import json
import sys
from collections import defaultdict

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Read from Fig. 4 of Kubilius et al. 2019 (change in ImageNet top-1, % points vs 2/4/2).
PAPER = {'1-1-1': -7, '2-2-2': -1, '2-4-2': 0, '4-4-4': 0}
ORDER = ['1-1-1', '2-2-2', '2-4-2', '4-4-4']

runs = defaultdict(list)
for f in glob.glob((sys.argv[1] if len(sys.argv) > 1 else 'runs') + '/cornet_s_*.json'):
    r = json.load(open(f))
    runs['-'.join(map(str, r['times']))].append(r['final_top1'])

base = np.mean(runs['2-4-2'])
rows = []
for k in ORDER:
    if k not in runs:
        continue
    v = np.array(runs[k])
    rows.append((k, len(v), v.mean(), v.std(ddof=1) if len(v) > 1 else float('nan'), v.mean() - base, PAPER[k]))

print(f"{'config':8s} {'n':>2s} {'top1':>7s} {'sd':>6s} {'delta':>7s} {'paper':>6s}")
for k, n, m, s, d, p in rows:
    print(f'{k:8s} {n:2d} {m:7.2f} {s:6.2f} {d:7.2f} {p:6.1f}')
with open('results_table.csv', 'w') as f:
    f.write('config,n_seeds,top1_mean,top1_sd,delta_vs_2-4-2,paper_delta\n')
    for r in rows:
        f.write(','.join(map(str, r)) + '\n')

x = np.arange(len(rows))
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(x, [r[4] for r in rows], yerr=[0 if np.isnan(r[3]) else r[3] for r in rows],
            fmt='o-', capsize=3, label='this reproduction')
ax.plot(x, [r[5] for r in rows], 's--', label='paper, Fig. 4 (read from figure)')
ax.set_xticks(x); ax.set_xticklabels([r[0].replace('-', '/') for r in rows])
ax.set_xlabel('recurrent steps in V2 / V4 / IT'); ax.set_ylabel('change in ImageNet top-1 (% pts vs 2/4/2)')
ax.axhline(0, color='gray', lw=0.5); ax.legend(); fig.tight_layout(); fig.savefig('recurrence_ablation.png', dpi=200)

In [ ]:
%%writefile /content/code/prep_tiny_imagenet.py
"""Download Tiny-ImageNet-200 (64x64, 200 classes, 100k train / 10k val) and put the
validation images into class folders so train.py can read it with ImageFolder.
Usage: python prep_tiny_imagenet.py /content/data
"""
import os
import shutil
import sys
import urllib.request
import zipfile

URL = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
root = sys.argv[1] if len(sys.argv) > 1 else 'data'
os.makedirs(root, exist_ok=True)
zpath = os.path.join(root, 'tiny-imagenet-200.zip')
ds = os.path.join(root, 'tiny-imagenet-200')

if not os.path.isdir(ds):
    if not os.path.exists(zpath):
        print('downloading', URL); urllib.request.urlretrieve(URL, zpath)
    print('extracting'); zipfile.ZipFile(zpath).extractall(root)

# train/<wnid>/images/*.JPEG  ->  train/<wnid>/*.JPEG
for wnid in os.listdir(f'{ds}/train'):
    img_dir = f'{ds}/train/{wnid}/images'
    if os.path.isdir(img_dir):
        for f in os.listdir(img_dir):
            shutil.move(f'{img_dir}/{f}', f'{ds}/train/{wnid}/{f}')
        shutil.rmtree(img_dir)

# val/images/*.JPEG + val_annotations.txt -> val/<wnid>/*.JPEG
ann = f'{ds}/val/val_annotations.txt'
if os.path.exists(ann):
    for line in open(ann):
        fname, wnid = line.split('\t')[:2]
        os.makedirs(f'{ds}/val/{wnid}', exist_ok=True)
        shutil.move(f'{ds}/val/images/{fname}', f'{ds}/val/{wnid}/{fname}')
    shutil.rmtree(f'{ds}/val/images'); os.remove(ann)
print('ready:', ds, '| train classes:', len(os.listdir(f'{ds}/train')), '| val classes:', len(os.listdir(f'{ds}/val')))

In [ ]:
%%writefile /content/code/check_against_official.py
"""Numerically check cornet_s.py against the official implementation (needs internet).
    pip install git+https://github.com/dicarlolab/CORnet
    python check_against_official.py
Loads the official pretrained weights into BOTH models and compares outputs in eval mode.
"""
import torch

from cornet_s import CORnet_S

import cornet  # official package

off = cornet.cornet_s(pretrained=True, map_location='cpu')
off = getattr(off, 'module', off).eval()
mine = CORnet_S((2, 4, 2))
sd = {k.replace('module.', '', 1): v for k, v in off.state_dict().items()}
print(mine.load_state_dict(sd, strict=True))   # raises on any key/shape mismatch
mine.eval()

torch.manual_seed(0)
x = torch.randn(4, 3, 224, 224)
with torch.no_grad():
    a, b = off(x), mine(x)
print('max abs diff:', (a - b).abs().max().item(), '| same argmax:', bool((a.argmax(1) == b.argmax(1)).all()))
assert torch.allclose(a, b, atol=1e-4), 'outputs differ: re-implementation is NOT equivalent'
print('OK: re-implementation matches official CORnet-S')

## 2. Validate the re-implementation against the official CORnet-S
Loads the official pretrained weights into both models and compares outputs. **If this fails, stop and fix `cornet_s.py` before training.**
(If `pip` can't find the package or the official API differs, check github.com/dicarlolab/cornet.)

In [ ]:
!pip -q install git+https://github.com/dicarlolab/CORnet
%cd /content/code
!python cornet_s.py
!python check_against_official.py

## 3. Data: Tiny-ImageNet to Colab's local disk (much faster than reading from Drive)

In [ ]:
%cd /content/code
!python prep_tiny_imagenet.py /content/data

## 4. Timing test (1 epoch of the baseline), then decide epochs/seeds
Multiply the time by 43 epochs × 4 configs × number of seeds to see if it fits your Colab budget.

In [ ]:
%cd /content/code
!python train.py --data /content/data/tiny-imagenet-200 --img-size 64 --min-scale 0.35 --workers 2 --amp \
    --epochs 1 --out /content/timing --name timing

## 5. Run the ablation (2/4/2, 1/1/1, 2/2/2, 4/4/4), checkpointed to Drive
Edit `SEEDS` in `run_all.py` for more seeds (differences of ~1 point need 2–3 seeds to mean anything).
If time-limited, add the same `--epochs 30 --step-size 14` to **every** config; never mix protocols across variants.

In [ ]:
%cd /content/code
!python run_all.py --data /content/data/tiny-imagenet-200 --img-size 64 --min-scale 0.35 \
    --workers 2 --amp --ckpt-every 2 --out {RUNS}

## 6. Results

In [ ]:
%cd /content/code
!python plot_results.py {RUNS}
from IPython.display import Image; Image('recurrence_ablation.png')

In [ ]:
import shutil
for f in ['results_table.csv', 'recurrence_ablation.png']:
    shutil.copy(f'/content/code/{f}', f'{DRIVE}/{f}')   # keep results in Drive